# COMP545 RAG Project – Outline

This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline and evaluation framework.

High-level stages:

1. Config & environment.
2. Data loading (queries + corpus).
3. Corpus → chunks → embeddings → Chroma (baseline index).
4. LLM setup
5. Logic graph construction.
6. Retrieval modules:
   - baseline_retrieve
   - graph_retrieve
7. RAG answerers:
   - answer_query_baseline
   - answer_query_graph
8. Evaluators (retrieval & generation).

## 1. Environment Setup & Dependencies


In [ ]:
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
import pandas as pd
import torch
import transformers
from PyPDF2 import PdfReader
import pdfplumber
import networkx as nx

# LangChain
from langchain.llms import HuggingFacePipeline
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA

# ==== Paths ====
QUERIES_PATH = "/data/queries.json"
PDF_PATH     = "/data/book.pdf"
CHROMA_DIR   = "chroma_db"

# cuda
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


## 2. (PDF → Sections)
- Corpus: `book.pdf`
- Queries: `queries.json` with:
  - `query_id`
  - `question`


In [ ]:
df_queries = pd.read_json(QUERIES_PATH)
print(df_queries.head())
print(f"Loaded {len(df_queries)} queries.")

In [ ]:
def load_book_with_page_overlap(pdf_path: str, overlap_size: int = 0):
    """
    Open the PDF, and for each page, 
    1. extract text
    2. Combine a few words from the end of the previous page with the current page(overlap)
    3. Returns a list of dicts: [{"page": int, "text": str}, ...].
    """
    sections = []

    def overlap_pages(page_text: str, next_page_text: str, overlap_size: int) -> str:
        page_words = page_text.split()
        next_page_words = next_page_text.split()
        overlap = page_words[-overlap_size:] if len(page_words) > overlap_size else page_words
        return " ".join(overlap) + " " + " ".join(next_page_words)

    try:
        reader = PdfReader(pdf_path, strict=False)
        previous_page_text = None
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                text = text.strip()
                if previous_page_text:
                    combined_text = overlap_pages(previous_page_text, text, overlap_size)
                    sections.append({"page": page_num + 1, "text": combined_text})
                else:
                    sections.append({"page": page_num + 1, "text": text})
                previous_page_text = text
    except Exception as e:
        print(f"Error loading PDF with PyPDF2: {e}")
        print("Falling back to pdfplumber...")
        try:
            with pdfplumber.open(pdf_path) as pdf:
                previous_page_text = None
                for page_num, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text and text.strip():
                        text = text.strip()
                        if previous_page_text:
                            combined_text = overlap_pages(previous_page_text, text, overlap_size)
                            sections.append({"page": page_num + 1, "text": combined_text})
                        else:
                            sections.append({"page": page_num + 1, "text": text})
                        previous_page_text = text
        except Exception as fallback_error:
            print(f"Error loading PDF with pdfplumber: {fallback_error}")
            return []

    return sections

book_data = load_book_with_page_overlap(PDF_PATH, overlap_size=5)
print(f"Loaded {len(book_data)} sections (pages).")
for sec in book_data[:3]:
    print(f"Page {sec['page']} snippet: {sec['text'][:200]}...\n")


In [ ]:
# Load the corpus PDF

PDF_PATH = "/data/book.pdf"\

book_data = load_book_with_page_overlap(PDF_PATH, overlap_size=5)
print(f"Loaded {len(book_data)} sections (pages).")

# Quick sanity check
for i, section in enumerate(book_data[:3]):
    print(f"Page {section['page']} snippet: {section['text'][:200]}...\n")


## 3. Documents → Chunks → Embeddings → Chroma
1. Wrap each page section as a LangChain `Document`.
2. Chunk documents into smaller passages.
3. Embed each chunk.
4. Store everything in a Chroma vector store (baseline index).

In [ ]:
# 3.1 Convert sections to Documents
documents = [
    Document(
        page_content=section["text"],
        metadata={"page": section["page"]}
    )
    for section in book_data
]

print("Base documents:", len(documents))

# 3.2 Chunk documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

all_splits = text_splitter.split_documents(documents)
print("Chunks after splitting:", len(all_splits))

## 4. LLM Setup (Gemma or other)
We load a causal LLM (e.g., Gemma-2-2B-IT) with `transformers`, create a
`text-generation` pipeline, and wrap it as a LangChain `llm`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-2-2b-it"  #we could change any LLm we want!

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#create llm pipeline(tokenize prompt -> Send tensors to GPU-> Call model.generate(...)-> decode tokens back to text)
generation_pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
    max_length=4096,
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)
print("LLM ready.")


## 5. Logic Graph Construction

We build a **logic graph** on top of the chunks.
Design (first version):

- **Nodes**: chunk IDs (and optionally extra "concept" nodes later).
- **Node attributes**:
  - `page`: page number
  - `text`: chunk text
  - embedding vector (optional or stored externally)
- **Edges** (first simple design):
  - *Adjacency edges*: between consecutive chunks in the same page.
  - *Section / topic edges* (later): based on headings, keywords, or high cosine similarity.

Goal: 
- Expand from an initially retrieved chunk to its neighbors.
- Re-rank chunks based on graph connectivity.


In [ ]:
#TODO:
def build_logic_graph(chunks: List[Document]) -> nx.Graph:
    skip

## 6. Retrieval Modules
two retrieval functions:
1. **Baseline retrieval**:
   - Pure vector similarity search (Chroma).
2. **Graph-aware retrieval**:
   - Start from top-k chunks by similarity.
   - Expand over the logic graph (e.g., neighbors).
   - Re-rank combined candidate set.

We will compare these two in evaluation.


In [ ]:
# 6.1 Baseline retriever
baseline_retriever = vectordb.as_retriever(
    search_kwargs={"k": 5}  # can tune later
)

def retrieve_baseline(query: str, k: int = 5) -> List[Document]:
    docs = baseline_retriever.get_relevant_documents(query)
    return docs[:k]


In [ ]:
# 6.2 graph retriever
#TODO:
def graph_retriever():
    skip

## 7. RAG Answer Functions

two answer functions:
- `answer_query_baseline(query)` → uses baseline retrieval only.
- `answer_query_graph(query)` → uses graph-aware retrieval.

Both will:
- Retrieve chunks.
- Build a prompt with context + question.
- Call the LLM.
- Return answer + retrieved docs.


## 8. Retrieval Evaluator
baseline vs graph-aware retrieval:
- For a subset of queries with ground truth labels (relevant pages/chunks),
- Compute metrics:
  - recall@k, MRR, etc.
- Compare:
  - Baseline retrieval
  - Graph-aware retrieval
